<a href="https://colab.research.google.com/github/mail2venkie/fastbook/blob/temp/myclean/08_collab_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
Mounted at /content/gdrive


In [2]:
#hide
from fastbook import *

In [3]:
from fastai.vision.all import *
path = untar_data(URLs.ML_100k)
path.ls()

(#23) [Path('/root/.fastai/data/ml-100k/mku.sh'),Path('/root/.fastai/data/ml-100k/u.item'),Path('/root/.fastai/data/ml-100k/u.user'),Path('/root/.fastai/data/ml-100k/u.occupation'),Path('/root/.fastai/data/ml-100k/u3.base'),Path('/root/.fastai/data/ml-100k/u4.test'),Path('/root/.fastai/data/ml-100k/u.info'),Path('/root/.fastai/data/ml-100k/u5.base'),Path('/root/.fastai/data/ml-100k/u.genre'),Path('/root/.fastai/data/ml-100k/u.data'),Path('/root/.fastai/data/ml-100k/u2.base'),Path('/root/.fastai/data/ml-100k/u4.base'),Path('/root/.fastai/data/ml-100k/ua.base'),Path('/root/.fastai/data/ml-100k/u2.test'),Path('/root/.fastai/data/ml-100k/ub.base'),Path('/root/.fastai/data/ml-100k/u1.base'),Path('/root/.fastai/data/ml-100k/u3.test'),Path('/root/.fastai/data/ml-100k/ub.test'),Path('/root/.fastai/data/ml-100k/u5.test'),Path('/root/.fastai/data/ml-100k/allbut.pl')...]

My trials
** START **

In [4]:
ratings_df = pd.read_csv(path/'u.data', delimiter='\t',
                         header=None,
                         names=['userId','itemID', 'rating', 'timestamp'])


In [5]:
ratings_df.head()

,userId,itemID,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [7]:
movies_df = pd.read_csv(path/'u.item',
                        delimiter='|',
                        header=None,
                        usecols=(0,1),
                        names=['itemID','name'],
                        encoding='latin-1'
                        )
movies_df.head()

,itemID,name
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [8]:
ratings = ratings_df.merge(movies_df)

In [9]:
ratings.head()

,userId,itemID,rating,timestamp,name
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [14]:
from fastai.collab import *
dls = CollabDataLoaders.from_df(ratings, item_name='name', bs=64)
dls.show_batch()

,userId,name,rating
0,243,Breaking the Waves (1996),5
1,227,Murder at 1600 (1997),3
2,918,Henry V (1989),5
3,325,Wallace & Gromit: The Best of Aardman Animation (1996),5
4,204,"English Patient, The (1996)",3
5,742,Primal Fear (1996),4
6,647,Waterworld (1995),4
7,608,Forrest Gump (1994),4
8,452,Casablanca (1942),5
9,406,"Sting, The (1973)",5


In [17]:
n_users = len(dls.classes['userId'])
n_movies = len(dls.classes['name'])
n_users, n_movies

(944, 1665)

In [18]:
n_factors = 5
userFactors = torch.rand(n_users, n_factors)
movieFactors = torch.rand(n_movies, n_factors)

In [60]:
one_hot_3 = one_hot(3, 10).float()
one_hot_3

tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])

In [69]:
userFactors.shape

torch.Size([944, 5])

In [48]:
userFactors

tensor([[0.6687, 0.9384, 0.1911, 0.5104, 0.2603],
        [0.7051, 0.3627, 0.3734, 0.4974, 0.7413],
        [0.4917, 0.9297, 0.4936, 0.0798, 0.3601],
        ...,
        [0.2362, 0.5058, 0.5893, 0.1367, 0.8303],
        [0.8660, 0.1300, 0.8265, 0.2405, 0.3048],
        [0.2028, 0.8894, 0.3641, 0.2457, 0.9710]])

In [72]:
class DotProduct(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
    self.userFactors = Embedding(n_users, n_factors)
    self.movieFactors = Embedding(n_movies, n_factors)
    self.y_range = y_range

  def forward(self, x):
    users = self.userFactors(x[:,0])
    movies = self.movieFactors(x[:,1])
    return sigmoid_range((users * movies).sum(dim=1), *self.y_range)

In [73]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())

In [74]:
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.914240,0.990373,00:08
1,0.672741,0.954694,00:07
2,0.470657,0.951659,00:08
3,0.371788,0.955834,00:08
4,0.323740,0.954018,00:07


In [71]:
learn.summary()

DotProduct (Input shape: 64 x 2)
Layer (type)         Output Shape         Param #    Trainable 
                     64 x 50             
Embedding                                 47200      True      
Embedding                                 83250      True      
____________________________________________________________________________

Total params: 130,450
Total trainable params: 130,450
Total non-trainable params: 0

Optimizer used: <function Adam at 0x7a8d9307c220>
Loss function: FlattenedLoss of MSELoss()

Model unfrozen

Callbacks:
  - TrainEvalCallback
  - CastToTensor
  - Recorder
  - ProgressCallback

In [75]:
class DotProductBias(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
    self.user_factors = Embedding(n_users, n_factors)
    self.user_bias = Embedding (n_users, 1)
    self.movie_factors = Embedding(n_movies, n_factors)
    self.movie_bias = Embedding (n_movies, 1)
    self.y_range = y_range

  def forward(self, x):
    users = self.user_factors(x[:,0])
    movies = self.movie_factors(x[:,1])
    result = (users * movies).sum(dim=1, keepdim=True)
    result += self.user_bias(x[:,0]) + self.movie_bias(x[:,1])
    return sigmoid_range(result, *self.y_range)

In [76]:
model1 = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model1, loss_func=MSELossFlat())

In [78]:
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.336194,0.941700,00:09
1,0.370502,0.920761,00:08
2,0.345459,0.905047,00:09
3,0.297657,0.892510,00:09
4,0.281629,0.888252,00:08


In [89]:
movie_bias = learn.model.movie_bias


<generator object Module.buffers at 0x7a8d86817220>

In [90]:
def create_params(size):
  return nn.Parameter(torch.zeros(*size).normal_(0,0.01))

In [104]:
class DotProductBias(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
    super().__init__()
    self.user_factors = create_params([n_users, n_factors])
    self.user_bias = create_params ([n_users])
    self.movie_factors = create_params([n_movies, n_factors])
    self.movie_bias = create_params ([n_movies])
    self.y_range = y_range

  def forward(self, x):
    users = self.user_factors[x[:,0]]
    movies = self.movie_factors[x[:,1]]
    result = (users * movies).sum(dim=1)
    result += self.user_bias[x[:,0]] + self.movie_bias[x[:,1]]
    return sigmoid_range(result, *self.y_range)

In [105]:
model = DotProductBias(n_users, n_movies, 50)
learn2 = Learner(dls, model, loss_func=MSELossFlat())

In [106]:
learn2.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.881147,0.953716,00:09
1,0.643065,0.903983,00:08
2,0.528467,0.881006,00:09
3,0.453813,0.862277,00:09
4,0.426114,0.857546,00:08


In [113]:
movie_bias= learn2.model.movie_bias.squeeze()
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['name'][i] for i in idxs]


['Titanic (1997)',
 "Schindler's List (1993)",
 'As Good As It Gets (1997)',
 'Rear Window (1954)',
 'L.A. Confidential (1997)']